# 4. LLM Fine-Tuning (Latent Oddity Data)

**Objective**: Train the Large Language Model (LLM) using the dataset generated by our Continuous VAE + Post-Hoc Oddity Quantizer pipeline.
By fine-tuning the LLM on this specifically crafted dataset, we teach it to inject geometric reasoning into its generation process. The model learns to output `<latent_N>` Riemannian tokens—representing its "hidden thoughts" traversing the data manifold—before providing the final natural language answer.


## 4.1 Environment Setup and Repository Cloning

To ensure reproducibility, this section automates the setup of the working environment:
1. **Google Drive Integration:** Mounts your personal Drive to store persistent data (checkpoints and processed datasets).
2. **Project Structure:** Automatically creates a `DLAI` folder in your Drive.
3. **Dependency Management:** Installs the `uv` package manager and resolves all requirements defined in `pyproject.toml`.
4. **Source Code:** Clones the `llama` branch from our GitHub repository to provide access to the `src` module and configuration files.

**Note for Evaluators:** Please authorize the Google Drive mount when prompted to allow the notebook to save and retrieve project files.

In [ ]:
import os, sys

# 1. Mount Google Drive
# Evaluators will need to accept the pop-up to connect their Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Setup directories on Drive
# Create the DLAI folder if it doesn't exist on their Drive
DRIVE_PROJECT_PATH = "/content/drive/MyDrive/DLAI"
if not os.path.exists(DRIVE_PROJECT_PATH):
    os.makedirs(DRIVE_PROJECT_PATH, exist_ok=True)
    print(f"Created project folder at: {DRIVE_PROJECT_PATH}")

# 3. UV Installation
# We use UV for much faster dependency management than standard pip
!curl -LsSf https://astral.sh/uv/install.sh | sh
os.environ['PATH'] = f"{os.path.expanduser('~')}/.cargo/bin:" + os.environ['PATH']

# 4. Clone the Repository (Branch: llama)
# If the local folder doesn't exist, clone the specific branch
%cd /content
if not os.path.exists("DLAI"):
    !git clone --branch llama https://github.com/irene-30/DLAI.git
else:
    print("Repo already exists, pulling latest changes...")
    !git -C DLAI pull

# 5. Synchronize pyproject.toml
# Copy the pyproject.toml from the cloned repo to the Drive folder (if necessary)
# or vice versa, to ensure that UV reads the correct dependencies.
!cp /content/DLAI/pyproject.toml {DRIVE_PROJECT_PATH}/pyproject.toml

# 6. Install dependencies via pyproject.toml
# This command reads the .toml file and installs everything necessary
%cd /content/DLAI
!uv pip install -e . --system

# 7. Add to the system path to allow imports from 'src'
sys.path.append("/content/DLAI")
%cd /content

print("✅ Setup completed successfully!")

Mounted at /content/drive
downloading uv 0.11.11 x86_64-unknown-linux-gnu
installing to /usr/local/bin
  uv
  uvx
everything's installed!
/content
Cloning into 'DLAI'...
remote: Enumerating objects: 637, done.
remote: Counting objects: 100% (191/191), done.
remote: Compressing objects: 100% (191/191), done.
remote: Total 637 (delta 135), reused 0 (delta 0), pack-reused 446 (from 2)
Receiving objects: 100% (637/637), 280.11 KiB | 7.78 MiB/s, done.
Resolving deltas: 100% (370/370), done.
/content/DLAI
Using Python 3.12.13 environment at: /usr
Resolved 87 packages in 850ms
Prepared 3 packages in 1.66s
Installed 3 packages in 16ms
 + bitsandbytes==0.49.2
 + dlai-metamath==0.1.0 (from file:///content/DLAI)
 + trl==0.29.1
/content


In [ ]:
# --- 1. Dependencies and Environment Setup ---
%pip install -q -U "torchao>=0.16.0"

import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, TrainingArguments
from trl import SFTTrainer
from peft import LoraConfig
from transformers.trainer_utils import get_last_checkpoint

# Make sure DLAI utils are in path
import sys
sys.path.append("/content/DLAI")

from src.utils import get_llm_tokenizer, get_llm_model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Using device: {device}")

# --- Directories ---
PATH_PROCESSED_DATA_ODDITY = "/content/drive/MyDrive/DLAI/data/processed/metamath_assorted_oddity_uv.jsonl"
DRIVE_SAVE_DIR = "/content/drive/MyDrive/DLAI/experiments/llm_latent_oddity_finetune_uv/"
FINAL_OUTPUT_DIR = os.path.join(DRIVE_SAVE_DIR, "final_model")

os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

### Step 1: Load and Pre-Tokenize the Oddity Dataset
We load the processed JSONL file containing the text intertwined with our custom discrete tokens. Tokenizing the dataset entirely in advance prevents the Hugging Face Trainer from stalling during the training loop. We also duplicate the `input_ids` into the `labels` column, which is the standard setup for Causal Language Modeling: the model learns to autoregressively predict the next token, encompassing both the geometric reasoning path and the final numerical solution.

In [ ]:
# --- 2. Load & Pre-Tokenize Oddity Data ---
tokenizer = get_llm_tokenizer()

try:
    print(f"Loading Oddity dataset from: {PATH_PROCESSED_DATA_ODDITY}")
    raw_dataset = load_dataset("json", data_files=PATH_PROCESSED_DATA_ODDITY, split="train")
except FileNotFoundError:
    print(f"❌ Error: Could not find the file at {PATH_PROCESSED_DATA_ODDITY}. Ensure Notebook 3 completed successfully!")
    raise

def tokenize_function(examples):
    tokenized = tokenizer(
        examples["text"],
        max_length=256,
        padding="max_length",
        truncation=True,
    )
    # Copy input_ids into labels for Causal Language Modeling (predicting the next token)
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

print("Tokenizing dataset...")
tokenized_dataset = raw_dataset.map(
    tokenize_function,
    batched=True,
    num_proc=os.cpu_count(),
    remove_columns=["text"]
)

# Format for PyTorch (Trainer expects tensors)
tokenized_dataset.set_format("torch")
print(f"✅ Dataset tokenized. Total samples: {len(tokenized_dataset)}")

### Step 2: Load Base Model and Setup LoRA
We load the base Llama-3 architecture and apply Low-Rank Adaptation (LoRA). Since the base model has never seen our custom `<latent_N>` tokens, we must explicitly resize its embedding layer. LoRA then freezes the massive pre-trained weights and injects small, trainable adapter matrices into the attention layers, drastically cutting down VRAM requirements and allowing us to efficiently train on a free T4 GPU.

In [ ]:
# --- 3. Setup Model and Resize Vocab ---
LLM_MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"

print(f"Loading Base LLM: {LLM_MODEL_NAME}")
model = get_llm_model(LLM_MODEL_NAME, len(tokenizer)).to(device)

# Crucial: If new Riemannian tokens were added, we must resize the embedding matrix
if model.get_input_embeddings().weight.shape[0] != len(tokenizer):
    print(f"Resizing embeddings from {model.get_input_embeddings().weight.shape[0]} to {len(tokenizer)}")
    model.resize_token_embeddings(len(tokenizer))

# Disable internal cache for training to save VRAM
model.config.use_cache = False

# --- 4. LoRA Configuration (T4 Optimization) ---
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"], # Target attention layers only to save memory
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
print("✅ Base Model and LoRA Config ready.")

### Step 3: Execute Training with Hugging Face Trainer
We define our `TrainingArguments` with a memory-saving strategy. To bypass hardware limitations, we combine a micro-batch size of 1 with 16 gradient accumulation steps to simulate a robust training signal. Coupled with gradient checkpointing and 8-bit paged optimizers, this setup maximizes training stability and avoids Out-of-Memory (OOM) failures during long runs.

In [ ]:
# --- 5. Training Arguments (Optimized for 16GB VRAM GPUs like T4) ---
training_args = TrainingArguments(
    output_dir=DRIVE_SAVE_DIR,
    per_device_train_batch_size=1,       # Micro-batch (Prevents Out-of-Memory)
    gradient_accumulation_steps=16,      # Effective batch size = 16
    learning_rate=2e-4,                  # Standard learning rate for LoRA
    num_train_epochs=1,
    fp16=True if device.type == "cuda" else False, # Mixed precision
    gradient_checkpointing=True,         # Massive memory saver (trades compute for RAM)
    optim="paged_adamw_8bit",            # Moves optimizer states to CPU / Quantizes them
    max_grad_norm=0.3,                   # Gradient clipping for stability
    warmup_ratio=0.03,
    lr_scheduler_type="constant",
    save_strategy="steps",
    save_steps=50,                       # Save checkpoints every 50 steps
    save_total_limit=2,                  # Keep only the last 2 checkpoints to save Drive space
    logging_steps=10
)

# --- 6. Initialize Trainer ---
trainer = SFTTrainer(
    model=model,
    train_dataset=tokenized_dataset,
    peft_config=peft_config,
    args=training_args,
)

print("\n--- 🚀 Starting LLM Fine-Tuning ---")

# Check if a valid checkpoint folder exists to resume training
last_checkpoint = get_last_checkpoint(DRIVE_SAVE_DIR)

if last_checkpoint is not None:
    print(f"--- 🔄 Resuming from checkpoint: {last_checkpoint} ---")
else:
    print("--- 🆕 Starting fresh training (no checkpoint found) ---")

# --- 7. Run Training ---
trainer.train(resume_from_checkpoint=last_checkpoint)

# --- 8. Save Final Model ---
print(f"Saving final model adapters and tokenizer to: {FINAL_OUTPUT_DIR}")
trainer.save_model(FINAL_OUTPUT_DIR)
tokenizer.save_pretrained(FINAL_OUTPUT_DIR)

print("✅ Training complete! Final model successfully saved.")